In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from models.linear_probes import linear_probe_tuned
from models.feature_generation import build_feature_bank, extract_encoder,pool_features
from preprocessing.dataset import PipistrelleDataset
import pandas as pd
from evaluation.metrics import plot_comprehensive_boxplots
from pathlib import Path
import os
from evaluation.statistical_tests import perform_encoder_statistical_analysis

In [ ]:

#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "perch2"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = True)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'perch2-2-noecho-2.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "NLM_BEATs"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = True)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'NLM_BEATs-2-noecho-2.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

#Testing feature extraction (without echo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "effnetb0"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder,
                             filter_echo = True)

features,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)
print(len(features))
pooled_feats = pool_features(features,windows= True ,window_pooled = False, method ='mean',encoder = encoder)
print(pooled_feats.shape)
npz_file_path = 'AVEX-effnetb0-2-noecho-2.npz'
np.savez(str(npz_file_path), pooled_feats=pooled_feats, labels=labels)

In [ ]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
X_eff0 = np.load(path + "\\AVEX-effnetb0-2-noecho-2.npz")['pooled_feats']
X_NLM = np.load(path + "\\NLM_BEATs-2-noecho-2.npz")['pooled_feats']
X_per2 = np.load(path + "\\perch2-2-noecho-2.npz")['pooled_feats']
y = np.load(path + "\\perch2-2-noecho-2.npz")['labels']
label_names = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']

In [ ]:
X_eff0_pooled = pool_features(X_eff0, windows=False, window_pooled=True, method='mean',encoder="effnetb0")
X_NLM_pooled = pool_features(X_NLM, windows=False, window_pooled=True, method='mean',encoder="NLM_BEATs")
X_per2_pooled = pool_features(X_per2, windows=False, window_pooled=True, method='mean',encoder="perch2")

In [ ]:
print(X_eff0_pooled.shape)

In [ ]:
# Create a mask: True where the row does NOT match [0, 0, 0, 0, 1]
X_eff0_arr = np.array(X_eff0_pooled)
X_NLM_arr  = np.array(X_NLM_pooled)
X_per2_arr = np.array(X_per2_pooled)
y_arr      = np.array(y)
mask = ~np.all(y_arr == [0, 0, 0, 0, 1], axis=1)

# Apply the mask to filter out those indices
X_eff0_pooled = X_eff0_arr[mask] # Remove .tolist() if you prefer keeping them as arrays
X_NLM_pooled  = X_NLM_arr[mask]
X_per2_pooled = X_per2_arr[mask]
y             = y_arr[mask]

In [ ]:
print(X_eff0_pooled.shape)
print(y[:,:4].shape)

In [ ]:
eff0_all_results = linear_probe_tuned(X_eff0_pooled, y[:,:4], n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
per2_all_results =linear_probe_tuned(X_per2_pooled,y[:,:4], n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
NLM_all_results =linear_probe_tuned(X_NLM_pooled, y[:,:4], n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
label_names = ['Type A', 'Type B', 'Type C', 'Type D']

results_test = {
            'ESP-EffNetB0': eff0_all_results,
            'Perch 2.0': per2_all_results,
            'NatureLM-BEATs': NLM_all_results
        }

from evaluation.metrics import plot_comprehensive_boxplots_no_echo
plot_comprehensive_boxplots_no_echo(results_test, label_names=label_names)

In [ ]:
df_stat = perform_encoder_statistical_analysis(results_test, label_names=label_names)

In [ ]:
from evaluation.statistical_tests import perform_and_plot_nemenyi
perform_and_plot_nemenyi(results_test, label_names=label_names)

In [ ]:
from evaluation.tables import process_encoder_folds_to_table_no_echo
latex_code = process_encoder_folds_to_table_no_echo(results_test)
with open("encoder_table_no_echo.tex", "w", encoding="utf-8") as f:
    f.write(latex_code)

In [ ]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
X_eff0 = np.load(path + "\\X_features2_not_normalized.npy")
X_NLM = np.load(path + "\\X_features_NLM.npy")
X_per2 = np.load(path + "\\perch_features.npz")['features']
y = np.load(path + "\\Y_labels2_not_normalized.npy")
label_names = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']

In [ ]:
#pooled features
X_eff0_pooled = pool_features(X_eff0, windows=False, window_pooled=True, method='mean',encoder="effnetb0")
X_NLM_pooled = pool_features(X_NLM, windows=False, window_pooled=True, method='mean',encoder="NLM_BEATs")
X_per2_pooled = X_per2

In [ ]:
from evaluation.tables import evaluate_avex_multilabel
embedding_dict ={
        'perch': X_per2_pooled,
        'effnetb0': X_eff0_pooled,
        'naturelm-beats': X_NLM_pooled
    }
latex_code =evaluate_avex_multilabel(embedding_dict, y, label_name="tab:encoder_eval")
# 3. Or save it directly to a .tex snippet file to import into your main document
with open("unsupervised_metrics.tex", "w") as f:
    f.write(latex_code)

In [ ]:
eff0_all_results = linear_probe_tuned(X_eff0_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
eff0_all_results2 = linear_probe_tuned(X_eff0_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
per2_all_results =linear_probe_tuned(X_per2_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
NLM_all_results =linear_probe_tuned(X_NLM_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
label_names = ['Type A', 'Type B', 'Type C', 'Type D','Echo']

results_test = {
            'ESP-EffNetB0': eff0_all_results,
            'Perch 2.0': per2_all_results,
            'NatureLM-BEATs': NLM_all_results
        }

plot_comprehensive_boxplots(results_test, label_names=label_names)

In [ ]:
df_stat = perform_encoder_statistical_analysis(results_test, label_names=label_names)

In [ ]:
from evaluation.statistical_tests import perform_and_plot_nemenyi
perform_and_plot_nemenyi(results_test, label_names=label_names)

In [ ]:
from evaluation.tables import process_encoder_folds_to_table
latex_code = process_encoder_folds_to_table(results_test)
print(latex_code)

In [ ]:
with open("encoder_table.tex", "w", encoding="utf-8") as f:
    f.write(latex_code)

In [ ]:
from evaluation.statistical_tests import evaluate_and_plot_linear_probe_algorithms
evaluate_and_plot_linear_probe_algorithms(per2_all_results, label_names=label_names)